# SPX Anchor Version 2 Demo

Version 2 adds strike-level gamma concentration, 0DTE weighting, time-to-close decay, and first-30m / VWAP / realized-vol adjustments.

In [ ]:
from src.spx_anchor.config import SpxAnchorSettings
from src.spx_anchor.backtest import build_snapshot_panel, run_version_backtest
from src.spx_anchor.live import generate_versioned_forecast
from src.spx_anchor.reporting import render_markdown_summary
from src.spx_anchor.synthetic import make_synthetic_spx_dataset
import pandas as pd

In [ ]:
settings = SpxAnchorSettings(min_train_sessions=6, retrain_every_n_sessions=2, lookback_rows_for_ranks=140)
spot_bars, option_snapshots = make_synthetic_spx_dataset(sessions=10)
panel = build_snapshot_panel(spot_bars, option_snapshots, settings=settings)
results, summary = run_version_backtest(panel, version='v2', settings=settings)
summary

In [ ]:
sessions = panel['session_date'].drop_duplicates().sort_values()
train_panel = panel[panel['session_date'].isin(sessions[:-1])].copy()
last_session = sessions.iloc[-1]
live_bars = spot_bars[spot_bars.index.normalize() == last_session].copy()
live_bars = live_bars[live_bars.index <= live_bars.index[65]]
live_options = option_snapshots[(pd.to_datetime(option_snapshots['as_of']).dt.normalize() == last_session) & (pd.to_datetime(option_snapshots['as_of']) <= live_bars.index[-1])].copy()
forecast = generate_versioned_forecast(pd.concat([spot_bars[spot_bars.index.normalize() < last_session], live_bars]), live_options, version='v2', settings=settings, rank_history=train_panel)
print(render_markdown_summary(forecast))